# ATLAS tutorial: preprocessing GLOFAS river discharge data

This notebook preprocesses **GLOFAS river discharge** files downloaded with the ATLAS hydro download notebook.

It is designed as a step-by-step tutorial for users who are not Python experts. In most cases, users only need to edit the **Input parameters** section and then run the notebook from top to bottom.

## What this notebook does

1. Imports the required Python packages.
2. Defines the input and output folders using generic paths.
3. Loads the GLOFAS files from `../data/glofas/downloads/{COUNTRY}/`.
4. Standardises coordinates so that the dataset can be handled consistently.
5. Optionally clips the data to a country boundary using a shapefile. Source: https://www.naturalearthdata.com/
6. Saves the processed result as a NetCDF file.

## Expected folder structure

This notebook assumes that the GLOFAS download notebook has already saved the raw files here:

```text
../data/glofas/downloads/{COUNTRY}/
```

For example, if `COUNTRY = "ecuador"`, the notebook will read files from:

```text
../data/glofas/downloads/ecuador/
```

The processed file will be saved by default here:

```text
../data/glofas/processed/{COUNTRY}/
```

## Before running

Make sure the required packages are installed in your Python environment:

```bash
pip install xarray geopandas numpy rioxarray cfgrib netcdf4
```

The package `cfgrib` also requires the ECMWF ecCodes library to read GRIB files.


## Step 1. Import required packages

Run this cell first. These packages are used to read GLOFAS files, manage paths, clip the data geographically and save the final NetCDF file.


In [1]:
from pathlib import Path
import glob

import geopandas as gpd
import numpy as np
import rioxarray  # needed to enable the .rio accessor used for clipping
import xarray as xr


## Step 2. Input parameters

Edit only this section for normal use.

### Country

`COUNTRY` is used both to find the input folder and to create the output folder.

For example, if `COUNTRY = "ecuador"`, input files are expected in:

```text
../data/glofas/downloads/ecuador/
```

### Dates

`START_DATE` and `END_DATE` are used to name the processed output file. They should match the time period downloaded in the previous notebook.

### Paths

The paths below are intentionally generic and relative to the notebook location:

- `INPUT_DIR`: folder containing the raw GLOFAS files downloaded in the previous notebook.
- `OUTPUT_DIR`: folder where the processed NetCDF file will be saved.
- `SHAPEFILE_PATH`: optional shapefile used to clip the data to a country boundary.

If you do not want to clip the data, set `CLIP_TO_COUNTRY = False`.


In [2]:
# =========================
# COUNTRY AND DATES
# =========================

# Country name. Keep it lowercase for folders.
COUNTRY = "ecuador"

# Dates used in the output filename.
START_DATE = "1991-01"
END_DATE = "1991-12"

# Variable name used in the output filename.
VARIABLE = "river_discharge"


# =========================
# INPUT AND OUTPUT PATHS
# =========================

# Raw GLOFAS files downloaded with the hydro download notebook.
INPUT_DIR = Path("../data/glofas_downloads") / COUNTRY

# Folder where the processed NetCDF file will be saved.
OUTPUT_DIR = Path("../data/processed") / VARIABLE / COUNTRY

# Optional shapefile with country boundaries.
# The shapefile is expected to contain a column called NAME_EN.
SHAPEFILE_PATH = Path("../world_map/ne_50m_admin_0_countries.shp")


# =========================
# PREPROCESSING OPTIONS
# =========================

# Set to False if you want to process the full downloaded bounding box without country clipping.
CLIP_TO_COUNTRY = True

# Country name as written in the shapefile NAME_EN column.
# For most countries, COUNTRY.capitalize() is enough, e.g. "ecuador" -> "Ecuador".
COUNTRY_NAME_IN_SHAPEFILE = COUNTRY.capitalize()

# Set to True to replace an existing processed file.
OVERWRITE_EXISTING_FILE = False

## Step 3. Helper functions

Run this cell without editing it.

The functions below:

1. check that input files exist;
2. load the downloaded GLOFAS files;
3. standardise latitude and longitude coordinate names;
4. make longitude values consistent;
5. optionally clip the data using a country shapefile;
6. save the final processed file.


In [3]:
def preprocess_reanalysis(dataset):
    """
    Standardise latitude and longitude precision before combining files.
    """
    if "longitude" in dataset.coords:
        dataset["longitude"] = np.round(dataset.longitude, 3)
    if "latitude" in dataset.coords:
        dataset["latitude"] = np.round(dataset.latitude, 3)
    return dataset


def find_glofas_files(input_dir):
    """
    Find GLOFAS files in the input folder.

    The download notebook usually saves GRIB files, but this function also accepts
    NetCDF files to make the preprocessing workflow easier to reuse.
    """
    input_dir = Path(input_dir)

    patterns = ["*.grib", "*.grb", "*.grib2", "*.nc"]
    files = []
    for pattern in patterns:
        files.extend(sorted(input_dir.glob(pattern)))

    if not files:
        raise FileNotFoundError(
            f"No GLOFAS files found in {input_dir}. "
            "Check COUNTRY and INPUT_DIR, and make sure the download notebook has completed."
        )

    return files


def load_hydro_data(input_dir):
    """
    Load all GLOFAS files from the input folder as one xarray dataset.
    """
    files = find_glofas_files(input_dir)

    print(f"Found {len(files)} file(s) in {input_dir}:")
    for file in files:
        print(f"- {file.name}")

    suffixes = {file.suffix.lower() for file in files}

    try:
        if suffixes.issubset({".grib", ".grb", ".grib2"}):
            dataset = xr.open_mfdataset(
                [str(file) for file in files],
                preprocess=preprocess_reanalysis,
                engine="cfgrib",
                combine="by_coords",
            )
        elif suffixes.issubset({".nc"}):
            dataset = xr.open_mfdataset(
                [str(file) for file in files],
                preprocess=preprocess_reanalysis,
                combine="by_coords",
            )
        else:
            raise ValueError(
                "The input folder contains mixed file formats. "
                "Keep only GRIB files or only NetCDF files in the selected folder."
            )
    except Exception as error:
        raise RuntimeError(
            "The files were found, but xarray could not open them. "
            "If they are GRIB files, check that cfgrib and ecCodes are correctly installed."
        ) from error

    return dataset


def rename_coordinates(dataset):
    """
    Rename common coordinate names to longitude and latitude.
    """
    rename_map = {}

    if "x" in dataset.coords:
        rename_map["x"] = "longitude"
    if "y" in dataset.coords:
        rename_map["y"] = "latitude"
    if "lon" in dataset.coords:
        rename_map["lon"] = "longitude"
    if "lat" in dataset.coords:
        rename_map["lat"] = "latitude"

    if rename_map:
        dataset = dataset.rename(rename_map)

    return dataset


def roll_longitudes(dataset):
    """
    Convert longitudes from 0/360 to -180/180 when needed.
    """
    dataset = dataset.assign_coords(
        longitude=((dataset.longitude + 180) % 360) - 180
    )
    dataset = dataset.sortby("longitude")
    return dataset


def fix_coordinates(dataset):
    """
    Standardise coordinate names, remove unnecessary scalar coordinates and fix longitude range.
    """
    dataset = rename_coordinates(dataset)

    if "spatial_ref" in dataset.coords:
        dataset = dataset.drop_vars("spatial_ref")

    dataset = dataset.squeeze(drop=True)

    if "latitude" not in dataset.coords or "longitude" not in dataset.coords:
        raise ValueError("The dataset must contain latitude and longitude coordinates.")

    dataset["latitude"] = dataset.latitude.astype("float32")
    dataset["longitude"] = dataset.longitude.astype("float32")

    if not bool((dataset.longitude < 0).any()):
        dataset = roll_longitudes(dataset)

    return dataset


def get_country_geometry(country_name, shapefile_path):
    """
    Read the country geometry from a shapefile.
    """
    shapefile_path = Path(shapefile_path)

    if not shapefile_path.exists():
        raise FileNotFoundError(
            f"Shapefile not found: {shapefile_path}. "
            "Set CLIP_TO_COUNTRY = False or update SHAPEFILE_PATH."
        )

    world = gpd.read_file(shapefile_path)

    if "NAME_EN" not in world.columns:
        raise ValueError("The shapefile must contain a NAME_EN column.")

    geometry = world.loc[world["NAME_EN"] == country_name, "geometry"]

    if geometry.empty:
        available_examples = ", ".join(sorted(world["NAME_EN"].dropna().unique())[:10])
        raise ValueError(
            f"Country '{country_name}' not found in shapefile NAME_EN column. "
            f"Examples of available names: {available_examples} ..."
        )

    return geometry


def clip_to_country(dataset, country_geometry):
    """
    Clip an xarray dataset to the selected country boundary.
    """
    dataset = fix_coordinates(dataset)
    dataset = dataset.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    dataset = dataset.rio.write_crs("EPSG:4326", inplace=False)
    return dataset.rio.clip(country_geometry, all_touched=True)


def drop_unneeded_variables(dataset):
    """
    Drop technical coordinates often present in GLOFAS GRIB files.
    """
    names_to_drop = [
        name for name in ["spatial_ref", "step", "surface", "valid_time"]
        if name in dataset.variables or name in dataset.coords
    ]

    if names_to_drop:
        dataset = dataset.drop_vars(names_to_drop, errors="ignore")

    return dataset


def build_output_filename(output_dir, variable, start_date, end_date):
    """
    Build the output NetCDF filename.
    """
    output_dir = Path(output_dir)
    filename = f"{variable}_{start_date}_{end_date}_processed.nc"
    return output_dir / filename


def preprocess_glofas_data(
    input_dir,
    output_dir,
    variable,
    start_date,
    end_date,
    clip_to_country_flag=True,
    country_name_in_shapefile=None,
    shapefile_path=None,
    overwrite=False,
):
    """
    Complete preprocessing workflow.
    """
    dataset = load_hydro_data(input_dir)
    dataset = fix_coordinates(dataset)

    if clip_to_country_flag:
        if country_name_in_shapefile is None or shapefile_path is None:
            raise ValueError(
                "country_name_in_shapefile and shapefile_path are required when clipping is enabled."
            )

        geometry = get_country_geometry(country_name_in_shapefile, shapefile_path)
        dataset = clip_to_country(dataset, geometry)
        print(f"Data clipped to: {country_name_in_shapefile}")
    else:
        print("Country clipping skipped. The full downloaded area will be saved.")

    dataset = drop_unneeded_variables(dataset)

    output_file = build_output_filename(output_dir, variable, start_date, end_date)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    if output_file.exists() and not overwrite:
        print(f"Output file already exists and overwrite is False: {output_file}")
        return dataset, output_file

    dataset.to_netcdf(output_file)
    print(f"Processed data written to: {output_file}")

    return dataset, output_file


## Step 4. Check the input folder

Run this cell before processing. It prints the folder that will be used and lists the files found inside it.

If this cell returns an error, check that the hydro download notebook saved the files in the expected folder.


In [4]:
print(f"Country: {COUNTRY}")
print(f"Input folder: {INPUT_DIR}")
print(f"Output folder: {OUTPUT_DIR}")

input_files = find_glofas_files(INPUT_DIR)
print(f"\nNumber of input files found: {len(input_files)}")
for file in input_files[:10]:
    print(f"- {file.name}")

if len(input_files) > 10:
    print(f"... and {len(input_files) - 10} more file(s)")


Country: ecuador
Input folder: ../data/glofas_downloads/ecuador
Output folder: ../data/processed/river_discharge/ecuador

Number of input files found: 1
- glofas_river_discharge_in_the_last_24_hours_ecuador_1991_01.grib


## Step 5. Run the preprocessing

Run this cell after checking the input parameters and the input folder.

The output will be a NetCDF file named like this:

```text
river_discharge_1991-01_2020-12_processed.nc
```

inside the selected `OUTPUT_DIR`.


In [5]:
processed_dataset, processed_file = preprocess_glofas_data(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    variable=VARIABLE,
    start_date=START_DATE,
    end_date=END_DATE,
    clip_to_country_flag=CLIP_TO_COUNTRY,
    country_name_in_shapefile=COUNTRY_NAME_IN_SHAPEFILE,
    shapefile_path=SHAPEFILE_PATH,
    overwrite=OVERWRITE_EXISTING_FILE,
)

processed_dataset


Found 1 file(s) in ../data/glofas_downloads/ecuador:
- glofas_river_discharge_in_the_last_24_hours_ecuador_1991_01.grib


/home/alessandrom/anaconda3/envs/preprocess_conda/lib/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(
ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/preprocess_conda/share/proj failed


Data clipped to: Ecuador
Processed data written to: ../data/processed/river_discharge/ecuador/river_discharge_1991-01_1991-01_processed.nc


<xarray.Dataset> Size: 5MB
Dimensions:    (time: 31, latitude: 130, longitude: 330)
Coordinates:
  * time       (time) datetime64[ns] 248B 1991-01-01 1991-01-02 ... 1991-01-31
  * latitude   (latitude) float32 520B 1.475 1.425 1.375 ... -4.925 -4.975
  * longitude  (longitude) float32 1kB -91.67 -91.62 -91.58 ... -75.27 -75.23
Data variables:
    dis24      (time, latitude, longitude) float32 5MB dask.array<chunksize=(31, 130, 330), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-12T10:02 GRIB to CDM+CF via cfgrib-0.9.1...

## Step 6. Quick output check

Run this optional cell to verify where the processed file was saved and inspect its size.


In [6]:
print(f"Processed file: {processed_file}")

if processed_file.exists():
    size_mb = processed_file.stat().st_size / (1024 * 1024)
    print(f"File size: {size_mb:.2f} MB")
else:
    print("The processed file was not created. Check the messages above.")


Processed file: ../data/processed/river_discharge/ecuador/river_discharge_1991-01_1991-01_processed.nc
File size: 5.10 MB


## Notes for adapting this notebook to a new country

To use this notebook for another country:

1. Set `COUNTRY` to the lowercase folder name used by the download notebook.
2. Make sure the raw files are in `../data/glofas/downloads/{COUNTRY}/`.
3. Check that `COUNTRY_NAME_IN_SHAPEFILE` matches the country name in the shapefile `NAME_EN` column.
4. Set `CLIP_TO_COUNTRY = False` if you do not have the shapefile or if the downloaded data are already clipped enough.
5. Run all cells from top to bottom.
